# Beer-game experiment v2 — training × reasoning effort

**Start here**

1. This version improves response checks and measurement. **It cannot guarantee significant results.**
2. **H1:** training increases supply-line weight β, averaged across reasoning conditions.
3. **H2_API:** among trained agents, low reasoning effort gives lower β than high effort. **This replaces the original 60/120-second manipulation; it does not test human time pressure.**
4. Four roles still form each game group. The first stage is a **separate diagnostic screen**, not a game.
5. Default settings are **offline + diagnostics**, with no key or payment needed. Set `MODE = "api"` for real GPT diagnostics. Only after reviewing those results set `STAGE = "game"`.

Keep this notebook beside `beer_game_v2.py`. The original notebook, completed results, and reports remain unchanged. The first paid diagnostic sends **64 requests**. A two-group-per-cell, 36-week game sends **1,184 requests**, including rules checks. Reasoning is billed as output and can cost more than the previous run. A local spending guard stops before the next request exceeds its estimated allowance; it is not an account billing limit.

**How to run:** Restart the kernel → Run All. For API mode paste the key into the hidden popup. Do not write the key in a cell. To resume after a spending stop, raise `BUDGET_USD`, keep other settings unchanged, and Run All. Never reuse the old v1 results folder.

In [1]:
# Uncomment only if dependencies are missing, then restart the kernel.
# %pip install "openai>=2,<3" "pydantic>=2.8,<3" numpy pandas scipy matplotlib ipykernel
from pathlib import Path
import sys, importlib, os, getpass

BASE = Path.cwd()
if (BASE / "Project" / "beer_game_experiment").is_dir():
    BASE = BASE / "Project" / "beer_game_experiment"
elif (BASE / "beer_game_experiment").is_dir():
    BASE = BASE / "beer_game_experiment"
if not (BASE / "beer_game_v2.py").exists():
    raise FileNotFoundError("Open this notebook in the beer_game_experiment folder, beside beer_game_v2.py.")
sys.path.insert(0, str(BASE))
import beer_game_v2 as study
importlib.reload(study)
from IPython.display import display, Markdown


## 1. Settings — change these before running

`GROUPS_PER_CELL = 2` is a measurement pilot: 8 groups / 32 agents. It is not enough for the planned p-value analysis. There are 4 cells: control/low, trained/low, control/high, trained/high. Each block contains one group in each cell, with the same demand path. Different blocks receive different prespecified paths.

Use a fresh `REPLICATE_ID` for independent new data. Identical settings reuse successful cached responses. The model snapshot, prompts, code, and settings are recorded. A code or experimental setting change creates a new run; budget changes alone do not.

In [7]:
MODE = "api"                 # "offline" or "api"
STAGE = "diagnostics"            # "diagnostics" first; "game" after reviewing the screen
GROUPS_PER_CELL = 4               # Pilot only. Choose final sample size before confirmation.
WEEKS = 36
REPLICATE_ID = "v2-pilot-01"
SEED = 20260927
BUDGET_USD = 3.00                 # Local estimate guard for THIS run, not your account balance
MAX_OUTPUT_TOKENS = 8192          # Includes reasoning; same ceiling in all cells
REQUEST_TIMEOUT_S = 180           # Infrastructure timeout, NOT an experimental time treatment

CONFIG = dict(mode=MODE, stage=STAGE, model=study.MODEL,
    groups_per_cell=GROUPS_PER_CELL, weeks=WEEKS, replicate_id=REPLICATE_ID,
    seed=SEED, budget_usd=BUDGET_USD, max_output_tokens=MAX_OUTPUT_TOKENS,
    timeout_s=REQUEST_TIMEOUT_S)
requests_planned = 64 if STAGE == "diagnostics" else GROUPS_PER_CELL*16*(WEEKS+1)
print("Model:", study.MODEL)
print(f"{MODE.upper()} / {STAGE}: {requests_planned:,} planned requests; {BUDGET_USD:.2f} USD local allowance.")
print("API diagnostics are paid; offline mode sends zero requests.")
# Scenarios, not forecasts: cache discounts ignored. Game histories can be longer.
for output_tokens in (200, 1000, 3000):
    estimate = requests_planned*(2000*study.RATES['input']+output_tokens*study.RATES['output'])/1e6
    print(f"If average input=2,000 and total output={output_tokens:,} tokens/request: about ${estimate:.2f}.")


Model: gpt-5.4-mini-2026-03-17
API / diagnostics: 64 planned requests; 3.00 USD local allowance.
API diagnostics are paid; offline mode sends zero requests.
If average input=2,000 and total output=200 tokens/request: about $0.15.
If average input=2,000 and total output=1,000 tokens/request: about $0.38.
If average input=2,000 and total output=3,000 tokens/request: about $0.96.


## 2. What changed and what a “proper reply” means

- Same neutral game rules and structured response fields for all agents. Responses include an order and exact readbacks of the observed state. Copying demand is permitted; an order need not match our hypothesis to be valid.
- Trained agents get a supply-line lesson with worked examples and feedback. Controls get a similar-length record-keeping lesson. Neither sees H1, H2, β, or a desired effect.
- Four basic rules questions are checked in each role context. These standalone checks screen the prompt/model, not persistent human-like learning. They do not provide adaptive coaching or select only successful agents.
- A refusal, incomplete response, incorrect readback, or API failure stops the run and preserves the record. No fabricated orders, repeated sampling until correct, or automatic changes to token limits.
- `reasoning.effort` is low/high using the **same model**. Record reasoning tokens and latency to check the manipulation. The common timeout is only for infrastructure.
- Game demand varies across weeks to offer more information for β estimation. Its path is balanced across conditions and hidden from agents in advance. Diagnostic states are separate and never injected into the game.

[Official model capabilities/pricing](https://developers.openai.com/api/docs/models/gpt-5.4-mini) · [Reasoning controls](https://developers.openai.com/api/docs/guides/reasoning) · [Structured outputs](https://developers.openai.com/api/docs/guides/structured-outputs)

Inspect the exact materials below. After the pilot, freeze these before an independent confirmation run. Training examples are repeated in context each request; the API does not learn new model weights.

In [8]:
print("SHARED RULES\n", study.STANDARD)
print("\nCONTROL LESSON\n", study.CONTROL)
print("\nTRAINING LESSON\n", study.TRAINING)


SHARED RULES
 You manage one role in a four-role beer supply chain: retailer,
wholesaler, distributor, factory. Minimize total team holding and backlog costs
across repeated weeks. Holding costs 0.5 per unit per week; backlog costs 1.
Only your local state and your own past rounds are available. No communication
with other roles. Orders take one week upstream, shipping takes two weeks
downstream, and factory production takes three weeks. Capacity is unlimited.
Orders cannot be canceled. Inventory is available stock; backlog is unfilled
customer demand; net inventory = inventory - backlog. Supply line is everything
previously ordered but not yet received. The state is AFTER this week's receipts
and shipments, BEFORE your new order. Future demand and the final week are unknown.
Choose any nonnegative integer order to pursue the cost objective. There is no
required ordering strategy or target order. Return the requested structured fields.
The observed_* fields must copy the current observ

## 3. Run — the only cell that sends API requests

The progress bar counts completed responses, including rules checks, and shows how many remain. Requests run sequentially to limit budget exposure, but four game orders all use the same weekly snapshot before any are committed. Close or interrupt the cell to stop.

If the local spending guard stops, successful responses are cached; raising only `BUDGET_USD` allows resumption. If an actual request failed, preserve the failed pilot, fix the reported issue, and start a fresh replicate. Unknown failed-request usage is conservatively reserved. Changing settings is a new experiment, not a continuation of the old data.

In [9]:
results, run_dir = None, None
client = None
try:
    if MODE == "api":
        from openai import AsyncOpenAI
        key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key (hidden): ")
        if not key.strip():
            raise ValueError("An API key is required in API mode.")
        client = AsyncOpenAI(api_key=key.strip(), max_retries=0, timeout=REQUEST_TIMEOUT_S)
        del key
    results, run_dir = await study.run_study(BASE, CONFIG, client=client)
    print("Saved to:", run_dir)
finally:
    if client is not None:
        await client.close()


Saved to: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/results/v2_api_7774f231a26789fc


## 4. Conclusions and detailed outputs

**Diagnostics:** compare how orders change when supply line varies with inventory held fixed, and vice versa. These finite differences are sensitivity checks, not β estimates. Flat responses or extensive zero orders mean the structural β model may still be unsuitable. Do not choose prompts based on whether trained orders move in the desired direction.

**Game:** fit β only when identification checks pass. Preserve all fitted values and exclusion reasons. The report shows conclusions first; CSV files hold the details.

There are two planned comparisons: H1 averaged across effort, and H2_API low−high among trained groups. Analysis uses paired **blocks of four groups**, not individual weeks. Two-sided paired sign-randomization p-values assume exchangeability under the sharp null; Holm adjustment accounts for the two tests. No p-values are reported for offline data, fewer than 8 complete blocks, or after any planned block is lost to β nonidentification. This conservative rule prevents significance testing on a behavior-selected subset. The reporting floor of 8 is not a power justification.

The β model is unchanged from v1: forecast + α × (target − net inventory − β × supply line), clipped at zero. If α is near zero, β is not identifiable. Do not relax this check to get a p-value.

In [10]:
if results is None or run_dir is None:
    raise RuntimeError("Complete the run cell successfully before generating a report.")
report = study.make_report(results, run_dir, CONFIG)
display(Markdown(report))
print("Detailed files:")
for path in sorted(run_dir.glob("*.csv")):
    print(path.name)


# GPT AGENT PILOT: v2 results

## Conclusions

1. Completed 48 decision records with validated state readbacks; no replacement orders.
2. 0/4 condition screens copied demand in every diagnostic state.
3. H1 and H2 have not been tested. Review inventory/supply-line sensitivity before running the game.
4. Diagnostic states are separate hypothetical retailer decisions; they are not four-role game trajectories.

## Diagnostic results

```
 trained effort  mean_order_change_supply_line_4_to_36  mean_order_change_inventory_0_to_24  copies_demand_all_states  zero_orders  median_reasoning_tokens                                                                            note
   False   high                              -6.666667                                 -8.0                     False           10                   1137.0 Descriptive retailer sensitivity screen; not a beta estimate or hypothesis test
   False    low                              -5.333333                                 -6.0                     False            5                     98.0 Descriptive retailer sensitivity screen; not a beta estimate or hypothesis test
    True   high                              -6.666667                                 -7.0                     False           10                   1079.5 Descriptive retailer sensitivity screen; not a beta estimate or hypothesis test
    True    low                              -4.000000                                 -5.0                     False            6                    220.5 Descriptive retailer sensitivity screen; not a beta estimate or hypothesis test
```

## Reasoning manipulation check

```
        mean_reasoning_tokens  median_reasoning_tokens  median_latency_s
effort                                                                  
high              1625.666667                   1121.5          5.530433
low                190.666667                    160.0          1.608079
```
Reasoning tokens and latency describe resource use, not human thinking time. Overlapping low/high resource use weakens the mechanism interpretation.

## Spending and interpretation

Recorded estimated charges/reservations: $0.2395. Not an invoice or account balance.
The original pilot remains separate. This redesigned protocol cannot confirm effects in humans. Do not repeatedly change prompts, exclusions, sample size, or models until p < .05. Use the pilot to fix measurement, then freeze a new protocol and collect independent confirmation data.

Detailed files:
diagnostic_summary.csv
diagnostics.csv
effort_check.csv
rules_checks.csv


## 5. Plan sample size before a confirmation run

Larger samples can improve power **after** reliable measurement is demonstrated. The table below uses a normal approximation for 80% power and conservative two-sided α=.025 per hypothesis. The standardized effect is a meaningful block contrast divided by its standard deviation, not a target picked to match this pilot. It ignores estimation failure and small-sample/randomization-test discreteness; use simulation-based planning with a stable pilot before treating a number as final.

Do not add groups repeatedly while checking p-values. Choose a fixed sample, analysis, prompts, model, and stopping rules beforehand. A significant result in this protocol would be evidence about these GPT agents under these conditions, not students.

In [11]:
display(study.sample_size_scenarios())


,standardized_paired_effect,approximate_groups_per_cell,total_groups,agents
0,0.3,106,424,1696
1,0.5,39,156,624
2,0.8,15,60,240
3,1.0,10,40,160
